# Level 4 — Data Cleaning & Feature Engineering

**Dataset:** Synthetic insurance claims data, modeled loosely on the kind of records a small agency (like MIA) would handle — but deliberately messy: missing values, inconsistent text formatting, mixed date formats, duplicate rows, and outliers, all injected on purpose.

**What you'll practice:**
- Diagnosing data quality problems *before* fixing them (don't clean blind)
- Handling missing values with judgment, not a single blanket rule
- Standardizing inconsistent categorical text
- Parsing mixed date formats
- Detecting and handling duplicates
- Detecting outliers (IQR method)
- Feature engineering: turning raw columns into useful derived ones

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Build the messy dataset

Don't worry about reading every line of this cell closely — the point isn't how the mess was created, it's how you clean it up. Just run it.

In [2]:
np.random.seed(7)

n = 60

claim_types_clean = ["Motor", "Health", "Property", "Life"]
# Messy versions with inconsistent casing/spacing to simulate manual data entry
claim_type_variants = ["Motor", "motor ", "MOTOR", "Health", "health", " Health",
                        "Property", "property", "Life", "life "]

regions_clean = ["Dar es Salaam", "Mbeya", "Arusha", "Dodoma", "Mwanza"]
region_variants = ["Dar es Salaam", "dar es salaam", "DSM", "Mbeya", "mbeya",
                    "Arusha", "Dodoma", "Mwanza", "mwanza "]

date_formats = ["%Y-%m-%d", "%d/%m/%Y", "%d-%b-%Y"]

rows = []
for i in range(n):
    claim_amount = np.random.gamma(shape=2, scale=400000)  # TZS, right-skewed like real claims
    if np.random.rand() < 0.05:
        claim_amount *= 15  # inject a few extreme outliers

    date_val = pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(np.random.uniform(0, 500)))
    date_str = date_val.strftime(np.random.choice(date_formats))

    row = {
        "ClaimID": f"C{1000 + i}",
        "ClaimType": np.random.choice(claim_type_variants),
        "Region": np.random.choice(region_variants),
        "ClaimAmount_TZS": round(claim_amount, 2),
        "DateFiled": date_str,
        "CustomerAge": np.random.randint(18, 75)
    }
    rows.append(row)

df = pd.DataFrame(rows)

# Inject missing values in a few columns, unevenly
for col, frac in [("ClaimAmount_TZS", 0.08), ("CustomerAge", 0.1), ("Region", 0.05)]:
    missing_idx = df.sample(frac=frac, random_state=1).index
    df.loc[missing_idx, col] = np.nan

# Inject a handful of exact duplicate rows
dupes = df.sample(4, random_state=2)
df = pd.concat([df, dupes], ignore_index=True)

# Shuffle so duplicates aren't obviously adjacent
df = df.sample(frac=1, random_state=3).reset_index(drop=True)

df.head(10)

,ClaimID,ClaimType,Region,ClaimAmount_TZS,DateFiled,CustomerAge
0,C1006,MOTOR,Arusha,321905.29,30-Jul-2024,73.0
1,C1055,property,Dodoma,784945.49,2024-09-08,66.0
2,C1050,motor,Dar es Salaam,NaN,11/08/2024,NaN
3,C1036,Property,Arusha,1180779.08,2024-12-11,33.0
4,C1048,Motor,mbeya,NaN,2024-09-12,NaN
5,C1023,Property,dar es salaam,534216.97,2024-06-10,48.0
6,C1031,Health,Mwanza,329053.09,03-Dec-2024,40.0
7,C1035,health,Dodoma,280730.05,07/07/2024,27.0
8,C1025,health,Mwanza,878634.41,2024-08-26,38.0
9,C1013,motor,Arusha,337017.17,2024-09-05,39.0


## 2. Diagnose before you clean

The instinct is to jump straight to `.fillna()` and `.drop_duplicates()`. Resist it — first find out exactly what's wrong and how much of it there is. Cleaning blind risks silently deleting more data than you realize, or fixing the wrong thing.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ClaimID          64 non-null     object 
 1   ClaimType        64 non-null     object 
 2   Region           61 non-null     object 
 3   ClaimAmount_TZS  59 non-null     float64
 4   DateFiled        64 non-null     object 
 5   CustomerAge      58 non-null     float64
dtypes: float64(2), object(4)
memory usage: 3.1+ KB


In [4]:
# How much is missing, per column?
df.isna().sum()

ClaimID            0
ClaimType          0
Region             3
ClaimAmount_TZS    5
DateFiled          0
CustomerAge        6
dtype: int64

In [5]:
# What do the messy categorical values actually look like?
print(df["ClaimType"].unique())
print(df["Region"].unique())

[np.str_('MOTOR') np.str_('property') np.str_('motor ')
 np.str_('Property') np.str_('Motor') np.str_('Health') np.str_('health')
 np.str_('life ') np.str_(' Health') np.str_('Life')]
[np.str_('Arusha') np.str_('Dodoma') np.str_('Dar es Salaam')
 np.str_('mbeya') np.str_('dar es salaam') np.str_('Mwanza')
 np.str_('mwanza ') np.str_('DSM') np.str_('Mbeya') nan]


In [6]:
# Exact duplicate rows
df.duplicated().sum()

np.int64(4)

## 3. Standardize categorical text

`"Motor"`, `"motor "`, and `"MOTOR"` are the same real-world category, but pandas treats them as three different strings. `.str.strip()` removes leading/trailing whitespace, `.str.title()` normalizes casing.

In [7]:
df["ClaimType"] = df["ClaimType"].str.strip().str.title()
df["ClaimType"].unique()

array(['Motor', 'Property', 'Health', 'Life'], dtype=object)

Region needs a bit more than casing — `"DSM"` is an abbreviation for Dar es Salaam, not a formatting quirk, so `.str.title()` alone won't fix it. This needs an explicit mapping.

In [8]:
region_map = {
    "dar es salaam": "Dar es Salaam",
    "dsm": "Dar es Salaam",
    "mbeya": "Mbeya",
    "mwanza": "Mwanza",
    "arusha": "Arusha",
    "dodoma": "Dodoma",
}

# Normalize to lowercase+stripped first, then map — so we only need one entry per region, not every casing variant
df["Region"] = df["Region"].str.strip().str.lower().map(region_map)
df["Region"].unique()

array(['Arusha', 'Dodoma', 'Dar es Salaam', 'Mbeya', 'Mwanza', nan],
      dtype=object)

Notice `NaN` is still in there — `.map()` on a value that's already missing stays missing, which is correct behavior. We haven't touched actual missing data yet, only fixed the *representation* of values that do exist.

## 4. Parse the inconsistent dates

`DateFiled` was generated in three different formats to simulate different data sources getting merged together (a very common real cause of this problem). `pd.to_datetime` with `format="mixed"` can infer the format per-row.

In [9]:
df["DateFiled"] = pd.to_datetime(df["DateFiled"], format="mixed")
df["DateFiled"].head()

0   2024-07-30
1   2024-09-08
2   2024-11-08
3   2024-12-11
4   2024-09-12
Name: DateFiled, dtype: datetime64[ns]

## 5. Handle duplicates

In [10]:
print("Rows before:", len(df))
df = df.drop_duplicates()
print("Rows after:", len(df))

Rows before: 64
Rows after: 60


## 6. Handle missing values — with judgment, not one blanket rule

Different columns deserve different treatment depending on what the missingness means and what you're about to do with the column.

In [11]:
df.isna().sum()

ClaimID            0
ClaimType          0
Region             3
ClaimAmount_TZS    5
DateFiled          0
CustomerAge        6
dtype: int64

In [12]:
# ClaimAmount is the core value we care about — filling it with a guess would distort financial analysis.
# Better to drop rows missing this one, since we can't meaningfully impute money.
df = df.dropna(subset=["ClaimAmount_TZS"])

# CustomerAge: fine to fill with the median — age is stable, unlikely to be wildly wrong, and we want to keep the row.
df["CustomerAge"] = df["CustomerAge"].fillna(df["CustomerAge"].median())

# Region: no sensible way to guess this, but we don't want to lose the row just for a missing region.
# Label it explicitly rather than silently dropping or guessing.
df["Region"] = df["Region"].fillna("Unknown")

df.isna().sum()

ClaimID            0
ClaimType          0
Region             0
ClaimAmount_TZS    0
DateFiled          0
CustomerAge        0
dtype: int64

Three different columns, three different decisions: **drop** (ClaimAmount — can't fake money), **impute with median** (CustomerAge — stable, low-risk to estimate), **explicit "Unknown" label** (Region — preserves the row without pretending we know something we don't). This is the actual skill in this section — not the pandas syntax, but the judgment call per column.

## 7. Detect outliers (IQR method)

In [13]:
Q1 = df["ClaimAmount_TZS"].quantile(0.25)
Q3 = df["ClaimAmount_TZS"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["ClaimAmount_TZS"] < lower_bound) | (df["ClaimAmount_TZS"] > upper_bound)]
print(f"Normal range: {lower_bound:,.0f} to {upper_bound:,.0f} TZS")
print(f"Number of outlier claims: {len(outliers)}")
outliers[["ClaimID", "ClaimType", "ClaimAmount_TZS"]]

Normal range: -616,806 to 1,861,775 TZS
Number of outlier claims: 3


,ClaimID,ClaimType,ClaimAmount_TZS
21,C1047,Property,2383942.87
22,C1034,Health,2401178.67
49,C1038,Health,7740901.88


The IQR method flags anything more than 1.5×(Q3−Q1) beyond the 25th/75th percentile — a standard, distribution-agnostic rule of thumb. Note we're **not** automatically dropping these. A claim being unusually large might be a data entry error, or it might be a genuinely large legitimate claim — that distinction needs a human, or at least a business rule, not just a statistical cutoff.

## 8. Feature engineering

Turning raw columns into new ones that are more directly useful for analysis or modeling.

In [14]:
# Month filed — useful for spotting seasonal claim patterns
df["MonthFiled"] = df["DateFiled"].dt.month

# Age bracket — often more useful for grouping/analysis than raw age
df["AgeGroup"] = pd.cut(
    df["CustomerAge"],
    bins=[0, 25, 35, 50, 65, 100],
    labels=["18-25", "26-35", "36-50", "51-65", "65+"]
)

# Flag high-value claims using the outlier boundary computed above
df["IsHighValueClaim"] = df["ClaimAmount_TZS"] > upper_bound

df[["ClaimID", "ClaimType", "Region", "ClaimAmount_TZS", "MonthFiled", "AgeGroup", "IsHighValueClaim"]].head(10)

,ClaimID,ClaimType,Region,ClaimAmount_TZS,MonthFiled,AgeGroup,IsHighValueClaim
0,C1006,Motor,Arusha,321905.29,7,65+,False
1,C1055,Property,Dodoma,784945.49,9,65+,False
3,C1036,Property,Arusha,1180779.08,12,26-35,False
5,C1023,Property,Dar es Salaam,534216.97,6,36-50,False
6,C1031,Health,Mwanza,329053.09,12,36-50,False
7,C1035,Health,Dodoma,280730.05,7,26-35,False
8,C1025,Health,Mwanza,878634.41,8,36-50,False
9,C1013,Motor,Arusha,337017.17,9,36-50,False
10,C1028,Life,Mwanza,529677.29,1,51-65,False
11,C1012,Health,Mwanza,303418.76,4,36-50,False


## 9. Sanity check the cleaned data

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55 entries, 0 to 63
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ClaimID           55 non-null     object        
 1   ClaimType         55 non-null     object        
 2   Region            55 non-null     object        
 3   ClaimAmount_TZS   55 non-null     float64       
 4   DateFiled         55 non-null     datetime64[ns]
 5   CustomerAge       55 non-null     float64       
 6   MonthFiled        55 non-null     int32         
 7   AgeGroup          55 non-null     category      
 8   IsHighValueClaim  55 non-null     bool          
dtypes: bool(1), category(1), datetime64[ns](1), float64(2), int32(1), object(3)
memory usage: 3.5+ KB


In [16]:
df.groupby("ClaimType")["ClaimAmount_TZS"].agg(["count", "mean", "median"]).round(0)

,count,mean,median
ClaimType,,,
Health,18,1090800.0,544362.0
Life,13,617271.0,529677.0
Motor,10,463540.0,329461.0
Property,14,857816.0,818111.0


## Exercises

1. Find the average `ClaimAmount_TZS` per `Region` (excluding "Unknown"). Which region has the highest average claim?
2. Create a new column `IsRecent` that's `True` if `DateFiled` is in the second half of 2024 or later, `False` otherwise.
3. Recompute the IQR outlier bounds *separately per `ClaimType`* instead of across all claims combined (hint: `groupby` + a function, or compute bounds within each group). Does a claim that looked normal overall look like an outlier within its own claim type, or vice versa?
4. (Challenge) The `"Unknown"` region rows are currently included in every aggregation. Write code that reports what percentage of total claim value comes from rows with unknown region — is it small enough to safely ignore, or large enough that the missing region data is a real problem worth going back to fix at the source?

In [17]:
# Your exercise answers here
region_avg = (
    df[df["Region"] != "Unknown"]
    .groupby("Region")["ClaimAmount_TZS"]
    .mean()
    .sort_values(ascending=False)
)
region_avg

Region
Mbeya            1.263112e+06
Dar es Salaam    1.119234e+06
Dodoma           6.747480e+05
Mwanza           6.009681e+05
Arusha           5.117977e+05
Name: ClaimAmount_TZS, dtype: float64

In [18]:
df["IsRecent"] = df["DateFiled"] >= pd.Timestamp("2024-07-01")
df[["ClaimID", "DateFiled", "IsRecent"]].head()

,ClaimID,DateFiled,IsRecent
0,C1006,2024-07-30,True
1,C1055,2024-09-08,True
3,C1036,2024-12-11,True
5,C1023,2024-06-10,False
6,C1031,2024-12-03,True


In [19]:
def get_outlier_bounds(group):
    Q1 = group.quantile(0.25)
    Q3 = group.quantile(0.75)
    IQR = Q3 - Q1
    return pd.Series({"lower": Q1 - 1.5 * IQR, "upper": Q3 + 1.5 * IQR})

bounds_by_type = df.groupby("ClaimType")["ClaimAmount_TZS"].apply(get_outlier_bounds).unstack()
bounds_by_type

,lower,upper
ClaimType,,
Health,-1.056206e+06,2.524082e+06
Life,-2.737658e+05,1.475718e+06
Motor,-4.953115e+05,1.385988e+06
Property,-1.850634e+05,1.793483e+06


In [20]:
# Now flag outliers using each row's own ClaimType bounds
def is_outlier_per_type(row):
    b = bounds_by_type.loc[row["ClaimType"]]
    return row["ClaimAmount_TZS"] < b["lower"] or row["ClaimAmount_TZS"] > b["upper"]

df["IsOutlier_PerType"] = df.apply(is_outlier_per_type, axis=1)

# Compare to the global flag from Section 8
df[["ClaimID", "ClaimType", "ClaimAmount_TZS", "IsHighValueClaim", "IsOutlier_PerType"]][
    df["IsHighValueClaim"] != df["IsOutlier_PerType"]
]

,ClaimID,ClaimType,ClaimAmount_TZS,IsHighValueClaim,IsOutlier_PerType
22,C1034,Health,2401178.67,True,False


In [21]:
total_value = df["ClaimAmount_TZS"].sum()
unknown_value = df[df["Region"] == "Unknown"]["ClaimAmount_TZS"].sum()

pct_unknown = (unknown_value / total_value) * 100
print(f"Unknown-region claims: {pct_unknown:.1f}% of total claim value")

Unknown-region claims: 0.0% of total claim value
